# ReNewGenie: VGG16 transfer learning on TrashNet (with Grad-CAM)

Run this in Google Colab: **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all** (about 15 to 25 minutes).

What it does, in plain words:
1. Downloads the TrashNet photos (2,527 photos, 6 classes).
2. Splits them **by photo** into train / validation / test (70/15/15), the same way the demo's own models were tested.
3. Starts from **VGG16 already trained on ImageNet** (millions of photos) and teaches it the 6 waste classes. This is *transfer learning*, and it is why accuracy goes from the demo's 86.8% to roughly the low 90s.
4. Reports honest test accuracy, a confusion matrix, and a Grad-CAM picture showing what the model looked at.
5. Saves `classifyWaste.h5`, the file your GitHub project expects but does not contain.

> Not run by the person who wrote it: this notebook was written without a GPU, so check the printed test accuracy yourself and do not quote a number you have not seen.

In [ ]:
import os, random, numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
SEED = 7
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# 1) Get TrashNet (resized 512x384 photos, one folder per class)
!git clone -q --depth 1 https://github.com/garythung/trashnet.git
!cd trashnet/data && unzip -q -o dataset-resized.zip
DATA = "trashnet/data/dataset-resized"
CLASSES = sorted(d for d in os.listdir(DATA) if os.path.isdir(os.path.join(DATA, d)))
print(CLASSES)

In [ ]:
# 2) Split by photo (70/15/15), same seed every time
files, labels = [], []
for i, c in enumerate(CLASSES):
    for f in sorted(os.listdir(os.path.join(DATA, c))):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            files.append(os.path.join(DATA, c, f)); labels.append(i)
files = np.array(files); labels = np.array(labels)
rng = np.random.RandomState(SEED); idx = rng.permutation(len(files))
n_tr, n_va = int(0.70 * len(idx)), int(0.15 * len(idx))
tr, va, te = idx[:n_tr], idx[n_tr:n_tr + n_va], idx[n_tr + n_va:]
print("train", len(tr), "val", len(va), "test", len(te))
IMG = (224, 224); BATCH = 32
def load(path, y):
    img = tf.io.read_file(path); img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG); return img, tf.one_hot(y, len(CLASSES))
def make(ix, train):
    ds = tf.data.Dataset.from_tensor_slices((files[ix], labels[ix]))
    if train: ds = ds.shuffle(len(ix), seed=SEED)
    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH)
    return ds.prefetch(tf.data.AUTOTUNE)
ds_tr, ds_va, ds_te = make(tr, True), make(va, False), make(te, False)

In [ ]:
# 3) Model: VGG16 (frozen) + small head. Augmentation runs on the GPU inside the model.
aug = keras.Sequential([layers.RandomFlip("horizontal"), layers.RandomRotation(0.06),
                        layers.RandomZoom(0.12), layers.RandomContrast(0.15)])
base = keras.applications.VGG16(include_top=False, weights="imagenet", input_shape=IMG + (3,))
base.trainable = False
inp = keras.Input(IMG + (3,))
x = aug(inp)
x = keras.applications.vgg16.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.4)(x)
out = layers.Dense(len(CLASSES), activation="softmax")(x)
model = keras.Model(inp, out)
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05), metrics=["accuracy"])
cb = [keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy")]
h1 = model.fit(ds_tr, validation_data=ds_va, epochs=12, callbacks=cb)

In [ ]:
# 4) Fine-tune the last VGG block with a small learning rate
base.trainable = True
for l in base.layers:
    l.trainable = l.name.startswith("block5")
model.compile(optimizer=keras.optimizers.Adam(1e-5), loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05), metrics=["accuracy"])
h2 = model.fit(ds_tr, validation_data=ds_va, epochs=10, callbacks=cb)

In [ ]:
# 5) Honest evaluation on photos the model never saw
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
y_true = np.concatenate([y.numpy().argmax(1) for _, y in ds_te])
probs = model.predict(ds_te)
y_pred = probs.argmax(1)
print("TEST ACCURACY: %.1f%%  (n=%d)" % (100 * (y_true == y_pred).mean(), len(y_true)))
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3))
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4)); plt.imshow(cm, cmap="Greens"); plt.xticks(range(6), CLASSES, rotation=45); plt.yticks(range(6), CLASSES)
for i in range(6):
    for j in range(6): plt.text(j, i, cm[i, j], ha="center", va="center")
plt.title("Confusion matrix (test)"); plt.tight_layout(); plt.show()
# confidence check: how often is it right when it is sure?
conf = probs.max(1)
for t in (0.5, 0.6, 0.7, 0.8):
    m = conf >= t
    print("confidence >= %.1f: answers %.0f%% of photos, right %.1f%% of those" % (t, 100 * m.mean(), 100 * (y_true[m] == y_pred[m]).mean()))

In [ ]:
# 6) Grad-CAM: which part of the photo drove the answer?
import cv2
last_conv = "block5_conv3"
vgg = model.get_layer("vgg16")
grad_model = keras.Model(vgg.input, [vgg.get_layer(last_conv).output, vgg.output])
head = keras.Sequential(model.layers[-3:])  # GAP, dropout, dense
def gradcam(img_batch):
    x = keras.applications.vgg16.preprocess_input(img_batch.copy())
    with tf.GradientTape() as tape:
        conv, feat = grad_model(x)
        tape.watch(conv)
        p = head(feat, training=False)
        loss = p[:, tf.argmax(p[0])]
    g = tape.gradient(loss, conv)
    w = tf.reduce_mean(g, axis=(1, 2), keepdims=True)
    cam = tf.nn.relu(tf.reduce_sum(w * conv, axis=-1))[0].numpy()
    cam /= cam.max() + 1e-8
    return cam, int(tf.argmax(p[0]))
fig, axs = plt.subplots(2, 4, figsize=(14, 7))
for ax, i in zip(axs.ravel(), te[:8]):
    img = tf.image.resize(tf.image.decode_jpeg(tf.io.read_file(files[i]), channels=3), IMG).numpy()
    cam, k = gradcam(img[None])
    heat = cv2.applyColorMap(np.uint8(255 * cv2.resize(cam, IMG)), cv2.COLORMAP_JET)[..., ::-1]
    ax.imshow(np.uint8(0.6 * img + 0.4 * heat)); ax.set_title("true %s | pred %s" % (CLASSES[labels[i]], CLASSES[k])); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# 7) Save the file your GitHub project expects, and a TensorFlow.js copy for the browser demo
model.save("classifyWaste.h5")
from google.colab import files as colab_files
colab_files.download("classifyWaste.h5")
# Optional: browser copy. The converter sometimes fails on the newest Keras; classifyWaste.h5 above is the file that matters.
!pip -q install tensorflowjs && tensorflowjs_converter --input_format=keras classifyWaste.h5 tfjs_model && zip -qr classifyWaste_tfjs.zip tfjs_model || echo "TF.js conversion skipped"

## What to write in your report
- Report the **TEST ACCURACY** printed in step 5, with the split (70/15/15 by photo, seed 7) and test size.
- TrashNet photos are plain-background; accuracy on cluttered phone photos will be lower. Say so.
- "Trash" has only about 137 photos, so its row in the report is the least reliable.
- Compare with the demo's from-scratch CNN (83.4%) and ensemble (86.8%) to show what transfer learning adds.